# 1

In [17]:
using Pkg
Pkg.add("CSV")
Pkg.add("DataFrames")

using CSV
using DataFrames

   Resolving package versions...
  No Changes to `C:\Users\fcdav\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\fcdav\.julia\environments\v1.11\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\fcdav\.julia\environments\v1.11\Project.toml`
  No Changes to `C:\Users\fcdav\.julia\environments\v1.11\Manifest.toml`


In [27]:
data = CSV.read(raw"C:\Users\fcdav\OneDrive\CausalAI\High_Dimensional_Linear_Models\Julia\input\apartments.csv", DataFrame)


Row,id,price,month,area,type,rooms,centredistance,schooldistance,clinicdistance,postofficedistance,kindergartendistance,restaurantdistance,collegedistance,pharmacydistance,ownership,buildingmaterial,hasparkingspace,hasbalcony,haselevator,hassecurity,hasstorageroom
,String,Int64,Int64,Int64,String31?,Int64,Float64,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,String15,String15?,String3,String3,String3?,String3,String3
1,a01d82c9529f98a54d64b9e061c9a73b,1199999,1,105,apartmentBuilding,4,5.06,1.08,0.949,0.623,0.791,1.054,3.062,0.335,condominium,brick,no,yes,yes,no,no
2,8373aa373dbc3fe7ca3b7434166b8766,650000,1,73,tenement,3,3.24,0.275,0.672,0.367,0.246,0.3,1.857,0.28,condominium,brick,no,no,no,no,no
3,7d0c31d5409caab173571cce3dcdf702,590000,1,69,blockOfFlats,3,3.94,0.139,1.336,0.926,0.93,0.071,0.786,0.304,condominium,brick,no,yes,no,no,yes
4,3eaa36a59b9354206703b5f6b2f2ff1d,584999,1,42,blockOfFlats,2,5.19,0.209,1.533,0.201,0.319,0.157,2.722,0.257,condominium,missing,no,yes,no,no,no
5,027b30cebbc49faf3094421b741ddd56,363000,1,46,blockOfFlats,2,1.65,0.25,0.624,0.51,0.177,0.232,0.986,0.273,condominium,concreteSlab,no,no,no,no,yes
6,27437e173a8c37d5002c0bf69c848b7c,717000,1,81,tenement,3,1.76,0.083,0.257,0.294,0.151,0.102,0.256,0.301,condominium,brick,no,no,no,no,yes
7,39404087c054348c27522c74ca21a973,689000,1,70,blockOfFlats,3,3.35,0.269,1.364,0.247,0.122,0.388,1.589,0.194,cooperative,concreteSlab,no,yes,yes,no,yes
8,7a3b00507c086fcf83562f22bb5d01fc,790000,1,67,apartmentBuilding,3,1.61,0.501,0.973,0.713,0.067,0.052,0.988,0.215,condominium,brick,yes,yes,yes,no,no
9,24af7cc54099d7930535543827604a9d,560000,1,69,blockOfFlats,3,2.29,0.371,0.592,0.213,0.222,0.24,0.583,0.721,condominium,brick,no,yes,no,no,yes


In [29]:
# 1. Crear una variable area2 que sea el cuadrado de area
data.area2 = data.area.^2
println("Variable 'area2' creada.")

Variable 'area2' creada.


In [39]:
# 2. Convertir 'hasparkingspace', 'hasbalcony', 'haselevator', 'hassecurity', 'hasstorageroom' a dummy variables
binary_cols = [:hasparkingspace, :hasbalcony, :haselevator, :hassecurity, :hasstorageroom]
for col in binary_cols
    if col in names(data)  # Using names(data) in Julia to get column names
        # Handle missing values by replacing them with "no" and then convert yes/no to 1/0
        data[!, col] = [ismissing(x) ? 0 : (x == "yes" ? 1 : 0) for x in data[!, col]]
    end
end

println("Variables binarias convertidas (yes/no a 1/0).")





Variables binarias convertidas (yes/no a 1/0).


In [41]:
# 3. Para cada último dígito de area (0,...,9), crear una variable dummy
for i in 0:9
    data[!, Symbol("end_$(i)")] = [row.area % 10 == i ? 1 : 0 for row in eachrow(data)]
end
println("Variables dummy 'end_0' a 'end_9' creadas basadas en el último dígito de 'area'.")

println("\nPrimeras filas del DataFrame después de la limpieza:")
println(first(data, 5))


Variables dummy 'end_0' a 'end_9' creadas basadas en el último dígito de 'area'.

Primeras filas del DataFrame después de la limpieza:
5×32 DataFrame
 Row │ id                                price    month  area   type               rooms  centredistance  schooldistance  clinicdistance  postofficedistance  kindergartendistance  restaurantdistance  collegedistance  pharmacydistance  ownership    buildingmaterial  hasparkingspace  hasbalcony  haselevator  hassecurity  hasstorageroom  area2  end_0  end_1  end_2  end_3  end_4  end_5  end_6  end_7  end_8  end_9 
     │ String                            Int64    Int64  Int64  String31?          Int64  Float64         Float64?        Float64?        Float64?            Float64?              Float64?            Float64?         Float64?          String15     String15?         Int64            Int64       String3?     String3      String3         Int64  Int64  Int64  Int64  Int64  Int64  Int64  Int64  Int64  Int64  Int64 
─────┼────────────────

# 2

In [43]:
# --- 3b Linear model estimation (4 points) ---

println("\n--- Iniciando Sección 3b: Linear model estimation ---")

# Variables a incluir en el modelo
area_last_digit_dummies = ["end_$(i)" for i in 0:8] # Excluir end_9 como base
continuous_vars = ["area", "area2", "distance"]
binary_vars = ["hasparkingspace", "hasbalcony", "haselevator", "hassecurity", "hasstorageroom"]
categorical_vars = ["month", "type", "rooms", "ownership", "buildingmaterial"] # GLM las maneja como categóricas


--- Iniciando Sección 3b: Linear model estimation ---


5-element Vector{String}:
 "month"
 "type"
 "rooms"
 "ownership"
 "buildingmaterial"

In [47]:
# First, import the necessary package that provides the @formula macro
# In Julia, this would typically be from the GLM.jl package
using GLM  # Add this import at the top of your code
# First, check what variables are actually available in your dataset
println("Available variables in the dataset:")
println(names(data))  # This is the Julia way to get column names

# Then modify your model formula to use existing variables
# For example, if you have 'dist' instead of 'distance':
model_formula = @formula(y ~ dist + other_variables)  # Replace with your actual variables

println("\nRealizando regresión lineal estándar...")
model_full = lm(model_formula, data)
println("\nTabla de resumen de la regresión lineal estándar:")
println(coeftable(model_full))








# 1. Regresión lineal estándar
# Construir la fórmula del modelo
all_covariates = vcat(area_last_digit_dummies, continuous_vars, binary_vars, categorical_vars)
formula_str = "price ~ " * join(all_covariates, " + ")
model_formula = eval(Meta.parse("@formula($formula_str)")) # Convierte el string en una fórmula

FormulaTerm
Response:
  price(unknown)
Predictors:
  end_0(unknown)
  end_1(unknown)
  end_2(unknown)
  end_3(unknown)
  end_4(unknown)
  end_5(unknown)
  end_6(unknown)
  end_7(unknown)
  end_8(unknown)
  area(unknown)
  area2(unknown)
  distance(unknown)
  hasparkingspace(unknown)
  hasbalcony(unknown)
  haselevator(unknown)
  hassecurity(unknown)
  hasstorageroom(unknown)
  month(unknown)
  type(unknown)
  rooms(unknown)
  ownership(unknown)
  buildingmaterial(unknown)

In [53]:
# First, check what variables are actually available in your dataset
print("Available variables in the dataset:")
print(data.columns)  # If using pandas
# or
print(names(data))   # If using Julia

# Then modify your model formula to use existing variables
# For example, if you have 'dist' instead of 'distance':
model_formula = "y ~ dist + other_variables"  # Replace with your actual variables

println("\nRealizando regresión lineal estándar...")
model_full = lm(model_formula, data)
println("\nTabla de resumen de la regresión lineal estándar:")
println(coeftable(model_full))

Available variables in the dataset:

LoadError: ArgumentError: column name :columns not found in the data frame

In [55]:
# Comentar los resultados de 'end_0'
println("\nComentario sobre el coeficiente de 'end_0':")
end0_coeff_row = coeftable(model_full).cols[1][findfirst(x -> occursin("end_0", x), coeftable(model_full).rownms)]
println("El coeficiente para 'end_0' es: ", end0_coeff_row)
println("Este coeficiente representa la diferencia promedio en el precio de un apartamento cuyo área termina en 0, en comparación con la categoría base (área terminada en 9), manteniendo todas las demás variables constantes.")
println("Un coeficiente positivo sugeriría una prima, mientras que uno negativo sugeriría un descuento.")



Comentario sobre el coeficiente de 'end_0':


LoadError: UndefVarError: `model_full` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [57]:
# 2. Regresión por partialling-out para el parámetro asociado con end_0

println("\nRealizando regresión por partialling-out para 'end_0'...")

# Paso 1: Regresar 'price' sobre todas las covariables EXCEPTO 'end_0'
covariates_exo_end0 = filter(x -> x != "end_0", all_covariates)
formula_price_exo_end0_str = "price ~ " * join(covariates_exo_end0, " + ")
model_price_exo_end0_formula = eval(Meta.parse("@formula($formula_price_exo_end0_str)"))
model_price_exo_end0 = lm(model_price_exo_end0_formula, data)
data.residuals_price = residuals(model_price_exo_end0)


Realizando regresión por partialling-out para 'end_0'...


LoadError: ArgumentError: There isn't a variable called 'distance' in your data; the nearest names appear to be: 

In [59]:
# Paso 2: Regresar 'end_0' sobre todas las demás covariables (las mismas que en el paso 1)
formula_end0_exo_end0_str = "end_0 ~ " * join(covariates_exo_end0, " + ")
model_end0_exo_end0_formula = eval(Meta.parse("@formula($formula_end0_exo_end0_str)"))
model_end0_exo_end0 = lm(model_end0_exo_end0_formula, data)
data.residuals_end0 = residuals(model_end0_exo_end0)

LoadError: ArgumentError: There isn't a variable called 'distance' in your data; the nearest names appear to be: 

In [61]:
# Paso 3: Regresar los residuos de 'price' sobre los residuos de 'end_0'
model_partial = lm(@formula(residuals_price ~ residuals_end0), data)
println("\nTabla de resumen de la regresión por partialling-out (coeficiente para 'end_0'):")
println(coeftable(model_partial))

# Verificar que los coeficientes sean los mismos
println("\nVerificación de coeficientes:")
coeff_end0_full_model = coeftable(model_full).cols[1][findfirst(x -> occursin("end_0", x), coeftable(model_full).rownms)]
coeff_end0_partial_model = coeftable(model_partial).cols[1][1] # El único coeficiente aparte del intercepto

println("Coeficiente de 'end_0' en el modelo completo: ", coeff_end0_full_model)
println("Coeficiente de 'end_0' en el modelo por partialling-out: ", coeff_end0_partial_model)

if isapprox(coeff_end0_full_model, coeff_end0_partial_model, atol=1e-6)
    println("Los coeficientes son prácticamente idénticos, verificando la consistencia de ambos métodos.")
else
    println("Advertencia: Los coeficientes difieren. Revisar la implementación.")
end

LoadError: ArgumentError: There isn't a variable called 'residuals_price' in your data; the nearest names appear to be: 

# 3

In [64]:
# ==============================================================================
# Paso 1: Entrenar el modelo solo con apartamentos cuya área NO termina en 0
# ==============================================================================

println("=== Paso 1: Entrenando el modelo sin apartamentos con área terminada en 0 ===")

# Crear un DataFrame filtrado donde 'end_0' no es 1 (es decir, el área no termina en 0)
df_no_end0 = filter(row -> row.end_0 == 0, df)

# Definir las variables explicativas para el modelo.
# Si 'X_vars' incluye 'end_0', debemos quitarla para el modelo base.
# Esto asume que X_vars es una lista de Strings o Symbols
X_vars_filtered = [var for var in X_vars if var != :end_0 && var != "end_0"] # Asegúrate de manejar ambos tipos

=== Paso 1: Entrenando el modelo sin apartamentos con área terminada en 0 ===


LoadError: ArgumentError: column name :end_0 not found in the data frame

In [66]:
# Construir la fórmula para el modelo OLS
# Necesitamos unir los nombres de las variables con " + " para la fórmula.
# Si X_vars_filtered está vacío o solo contiene una variable, ajusta aquí.
if isempty(X_vars_filtered)
    formula_str = "price ~ 1" # Solo intercepto si no hay otras variables
else
    # Convertir a Symbol si no lo son ya, y luego a String para unir
    vars_for_formula = join([string(var) for var in X_vars_filtered], " + ")
    formula_str = "price ~ " * vars_for_formula
end

# Entrenar el modelo OLS con el DataFrame filtrado
form_filtered = @eval(@formula($(Meta.parse(formula_str)))) # Usar @eval y Meta.parse para construir la fórmula dinámicamente
model_no_end0 = lm(form_filtered, df_no_end0)

println("\nCoeficientes del Modelo OLS sin apartamentos con área terminada en 0:")
println(coeftable(model_no_end0))


LoadError: UndefVarError: `X_vars_filtered` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
# ==============================================================================
# Paso 2: Predecir precios para la muestra completa
# ==============================================================================

println("\n=== Paso 2: Prediciendo precios para la muestra completa ===")

# Asegúrate de que las columnas usadas en 'form_filtered' existan en el DataFrame original `df`.
# `predict` de GLM.jl es inteligente y usará las columnas correctas.
predicted_prices = predict(model_no_end0, df)

# Añadir las predicciones al DataFrame original `df`
df[!, :predicted_price] = predicted_prices

println("Precios predichos añadidos a 'df'.")

In [ ]:
# ==============================================================================
# Paso 3: Comparar promedios
# ==============================================================================

println("\n=== Paso 3: Comparando promedios para apartamentos con área terminada en 0 ===")

# Filtrar los apartamentos donde el área termina en 0
apartments_end0 = filter(row -> row.end_0 == 1, df)

# Verificar si hay apartamentos con end_0 == 1 para evitar errores en 'mean'
if isempty(apartments_end0)
    println("No hay apartamentos en el conjunto de datos cuya área termine en 0.")
else
    # Calcular el precio actual promedio para estos apartamentos
    average_actual_price_end0 = mean(apartments_end0.price)

    # Calcular el precio predicho promedio para estos apartamentos
    average_predicted_price_end0 = mean(apartments_end0.predicted_price)

    println(f"Precio actual promedio para apartamentos con área terminada en 0: {average_actual_price_end0:.2f}")
    println(f"Precio predicho promedio (por el modelo sin 'end_0') para apartamentos con área terminada en 0: {average_predicted_price_end0:.2f}")